# Topic: Tokenization Algorithms (BPE, WordPiece, SentencePiece, Special Tokens & OOV)

## Definition (30-second explanation)
*   Imagine building words out of LEGO bricks. Character-level tokenization gives you individual 1x1 studs (too slow to build a house), while whole-word tokenization gives you entire pre-molded rooms (you need infinite unique rooms for every word in the dictionary).
*   **Subword Tokenization (BPE, WordPiece, SentencePiece)** breaks text into common modular bricks (e.g., "un", "break", "able"), balancing vocabulary size with sequence length.
*   By combining modular subword pieces, the model can represent any unseen or rare word, completely eliminating the Out-of-Vocabulary (OOV) failure.

## Why Interviewers Ask This
*   Tokenization directly dictates LLM costs, inference latency, prompt chunking in RAG pipelines, and embedding drift.
*   Interviewers want to test if you know why LLMs fail at basic tasks like character counting, reversing strings, or arithmetic (tokens compress multi-digit numbers unpredictably).
*   It exposes whether you understand that neural networks do not process raw text, but discrete integer IDs produced by a standalone, statistical pre-processor.

## Core Concepts (The 3-Layer Anatomy)
*   **The Bottleneck:** Whole-word tokenizers require massive vocabulary tables (1M+ words) that bloat GPU VRAM via embedding matrices and crash on typos/new words (`<UNK>`). Character tokenizers make sequences 5x longer, blowing up Transformer $O(N^2)$ memory.
*   **The Mechanism:** 
    *   *BPE (Byte-Pair Encoding):* Starts at character or byte level and iteratively merges the most *frequently co-occurring* adjacent token pairs. (Used by GPT-2/3/4, Llama).
    *   *WordPiece:* Similar to BPE, but merges pairs that maximize the *training data likelihood* (scoring by $\frac{\text{Count}(AB)}{\text{Count}(A) \times \text{Count}(B)}$). (Used by BERT).
    *   *SentencePiece:* Treats the input as a raw stream of characters/bytes including whitespace (marked as `_`), removing language-dependent pre-tokenization rules. (Used by T5, Llama, Mistral).
*   **The Trade-off:** Larger vocabularies compress text into fewer tokens (lowering API cost and latency), but dramatically increase the model's initial embedding and final output projection matrix parameter footprints.

## When to Use
*   **Byte-level BPE / SentencePiece:** Standard for training or deploying modern multilingual LLMs and code models.
*   **Special Tokens (`<pad>`, `<eos>`, `<unk>`, `[CLS]`, `[SEP]`):** Use when structuring conversational chat templates, separating RAG documents, or masking inputs.

## Advantages
*   **Zero True OOV (Out-of-Vocabulary):** Byte-fallback mechanisms guarantee any unseen word, emoji, or symbol can be broken down into known subwords or raw UTF-8 bytes.
*   **Morphological Awareness:** Naturally captures prefixes and suffixes (e.g., "play", "playing", "player"), enabling parameter sharing across related word forms.
*   **Language Agnostic:** SentencePiece works seamlessly across non-segmented languages like Japanese, Chinese, or Thai without needing language-specific word splitters.

## Limitations
*   **Arithmetic & Spelling Blindspots:** "1845" might become one token while "1846" splits into two, making number manipulation unintuitive for the LLM.
*   **Tokenization Disparity (Cost Imbalance):** Non-English languages often require 2x to 5x more tokens to convey the exact same meaning, inflating API costs and latency for global users.

## Common Comparisons
*   **BPE vs. WordPiece:** BPE merges by raw pair frequency. WordPiece merges by statistical likelihood/mutual information score.
*   **BPE vs. SentencePiece:** Standard BPE relies on pre-tokenization (splitting by spaces first), which fails on languages without spaces. SentencePiece treats spaces as just another character (`_`), processing raw byte streams natively.

## Common Interview Traps
*   **Thinking tokenizers are trained with the LLM:** Tokenizers are trained *before* the neural network using purely statistical text algorithms, completely frozen, and run on the CPU.
*   **Modifying Special Tokens improperly:** Adding custom tokens (like `<|tool_call|>`) to an existing tokenizer without resizing the model's embedding matrix via `model.resize_token_embeddings(len(tokenizer))` will cause runtime CUDA crashes.

## Python Syntax (Hugging Face Transformers)
*   *Practical tokenization, decoding, and handling special chat tokens.*

```python
from transformers import AutoTokenizer

# 1. Load tokenizer for a modern model (e.g., Llama/Mistral family using SentencePiece/BPE)
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")

# 2. Tokenize text with explicit padding and truncation for batch inference
# WHY THIS API?: return_tensors='pt' returns PyTorch tensors; truncation prevents OOM errors
inputs = tokenizer(
    ["Hello, world!", "RAG pipelines require smart chunking."],
    padding=True,              # Pads shorter sequences with pad_token_id to ensure uniform tensor shape
    truncation=True,           # Cuts sequences exceeding max_length
    max_length=16,             # Hard cap on input length
    return_tensors="pt"
)

# 3. Inspect raw token IDs and decode back to text
token_ids = inputs["input_ids"][0]
print("Token IDs:", token_ids)
print("Decoded text:", tokenizer.decode(token_ids, skip_special_tokens=False))
```

## Important Formula
$$\text{WordPiece Score}(A, B) = \frac{\text{Count}(AB)}{\text{Count}(A) \times \text{Count}(B)}$$
*(BPE picks the highest $\text{Count}(AB)$; WordPiece normalizes by individual frequencies to prioritize pairs that appear together surprisingly often.)*

## 45-Second Interview Answer
"Tokenizers translate raw text into discrete token IDs before a Transformer ever sees them. Traditional whole-word tokenizers suffer from massive vocabularies and out-of-vocabulary `<UNK>` errors, while character tokenizers inflate sequence lengths beyond quadratic memory limits. Modern subword algorithms like BPE, WordPiece, and SentencePiece solve this by iteratively merging frequent subwords and falling back to bytes for rare words. This provides zero OOV failures and optimal context efficiency, though it introduces known quirks like arithmetic difficulties and multi-language cost disparities."

## Practice Questions:

### Q1: Tokenization & RAG Chunking Disparities
**Question:** What are the major failure modes of fixed character-count chunking (e.g., 1000 characters) in a RAG pipeline handling English, German, and Python code? How does subword tokenization behave across these modalities?

**Answer:**
"Fixed character chunking fails in two distinct ways:
1. **Semantic & Syntax Fragmentation:** It arbitrarily cuts sentences and functions in half. In Python, cutting an indentation or function block breaks the Abstract Syntax Tree (AST), ruining retrieval quality.
2. **Token Density Disparity (The Tokenizer Trap):** Embedding models have token context limits (e.g., 512 tokens), not character limits. Because subword tokenizers (BPE/SentencePiece) are trained predominantly on English text:
   * **English:** High compression (~4 chars/token). 1,000 characters $\approx$ 250 tokens.
   * **German:** Long compound words get fragmented into numerous tiny subwords. 1,000 characters might yield 500–700 tokens, risking silent truncation by the embedding model.
   * **Code:** Variable names (camelCase/snake_case) and whitespace create high token density. 

**Solution:** Always chunk using **token counts** via the actual tokenizer library (e.g., `tiktoken` or Hugging Face `AutoTokenizer`) and use syntax-aware splitters (like LangChain's `RecursiveCharacterTextSplitter` or Tree-sitter code splitters)."

**Interview Tips:**
* **Key Ratio:** Remember ~4 chars per token for English as a rule of thumb, but emphasize that non-English languages and code have significantly higher token density.
* **Silent Truncation:** Always highlight that embedding models truncate by tokens, so character-based chunking risks dropping critical context without throwing an error.